# **Generalización semántica del campo `profile_bio`**

Pipeline de 5 etapas que toma los `profile_bio` ya procesados con regex (con placeholders tipo `<LOCATION>`, `<PERSON>`, `<PHONE_NUMBER>`, `[USUARIO]`) y produce una versión **generalizada, anonimizada e irreversible en francés**, preservando señales útiles para inferir género y rango etario.

**Etapas:**
1. Normalización Unicode (estilizados → ASCII)
2. Detección de idioma + traducción a español si es necesario
3. Generalización con Gemma 3 12B (prompt género/edad-aware)
4. Validador SOFT (reporta, no bloquea)
5. Traducción ES → FR universal

**Entrada esperada:** `tweets_filtrados` (DataFrame con columna `profile_bio`).

**Salida:** mismo DataFrame con columna nueva `profile_bio_fr` + JSON de auditoría.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

tweets_user_id = pd.read_csv(
    '/content/drive/MyDrive/TFM/7. Riesgo_suicidio/input_categorizacion.csv'
)

tweets_user_id.head()

,user_id,tweets_concat,n_tweets
0,01fc52c4bc9ffc8c,[TWEET 1] Cusndo sera el día que yo diga “no m...,10
1,04d09936041bc786,"[TWEET 1] Mi padre tiene un tumor, le operan e...",2
2,05306b7fa750a5d8,[TWEET 1] Estoy muy resfriado y tengo chuchos ...,1
3,0532b08c30376fde,[TWEET 1] llevo todo el día en la cama\n\n[TWE...,32
4,0588fd52a2609179,[TWEET 1] RT [USUARIO]: Cuando tu propia famil...,4


In [3]:
tweets_psico = pd.read_csv(
    '/content/drive/MyDrive/TFM/8. Categorizacion/clasificacion_psicologas.csv'
)

tweets_psico.head()

,tweets_concat,clasif_modelo,clasif_experto
0,[TWEET 1] DEP...,Dudoso,Dudoso
1,[TWEET 1] RT [USUARIO]: Ya sé que no me lee na...,Dudoso,Dudoso
2,[TWEET 1] Stoy faking triste y a punto de puto...,Positivo,Dudoso
3,[TWEET 1] Cusndo sera el día que yo diga “no m...,Positivo,Positivo
4,[TWEET 1] Alguien me espía\n\n[TWEET 2] RT [US...,Dudoso,Dudoso


In [4]:
tweets_psico = pd.merge(
    tweets_psico,
    tweets_user_id[['tweets_concat', 'user_id']],
    on='tweets_concat',
    how='inner'
)

tweets_psico.shape

(73, 4)

In [5]:
tweets_agrupados = pd.read_csv(
    '/content/drive/MyDrive/TFM/positivo_control/Escritorio_remoto/agrupacion_usuarios.csv'
)

tweets_agrupados.head()

,user_id,userName,name,tweets_concat,n_tweets,first_tweet,last_tweet,profile_bio,location,followers,following,lang
0,003aa0c7bf89f9e3,USR_9b3b8ec2,[NOMBRE],[TWEET 1] RT [USUARIO]: Los inmigrantes regula...,93,2026-04-02,2026-04-18,Madre de familia y activista para la protecció...,<LOCATION>,337.0,1554.0,es
1,009c8bb24862d606,USR_0bb8dab2,[NOMBRE],"[TWEET 1] RT [USUARIO]: Esto es mi BCN, noche ...",99,2017-08-17,2026-03-27,NaN,Currently in <LOCATION> 🇪🇸,172.0,531.0,es
2,00bcbd0b067ef737,USR_0204aec3,[NOMBRE],"[TWEET 1] RT [USUARIO]: ‼️Alto y claro, <PERSO...",98,2026-04-19,2026-04-20,"Viajar, conocer lugares y su gente es mi pasión.","<LOCATION>, <LOCATION>",706.0,672.0,es
3,00fa95048919fd68,USR_8405c455,[NOMBRE],[TWEET 1] i would love to check my phone here\...,67,2024-10-06,2026-04-19,🔗🔗🔗🔗,en la silla de la tarta :$,125.0,458.0,en
4,011a3468f9d54417,USR_d78d028e,[NOMBRE],[TWEET 1] Mala Gente.....!!!!! [URL]\n\n[TWEET...,21,2023-01-03,2023-01-12,NaN,<PERSON>,1098.0,1128.0,es


In [6]:
cols_to_drop = [col for col in tweets_agrupados.columns if 'clasif_modelo' in col or 'clasif_experto' in col]
tweets_agrupados = tweets_agrupados.drop(columns=cols_to_drop, errors='ignore')

tweets_agrupados = pd.merge(
    tweets_agrupados,
    tweets_psico[['user_id', 'clasif_modelo', 'clasif_experto']],
    on='user_id',
    how='inner'
)

tweets_agrupados.head()

,user_id,userName,name,tweets_concat,n_tweets,first_tweet,last_tweet,profile_bio,location,followers,following,lang,clasif_modelo,clasif_experto
0,01fc52c4bc9ffc8c,USR_69f55ce3,[NOMBRE],[TWEET 1] Nacemos solos &amp; morimos en el al...,74,2025-07-22,2026-04-16,NaN,"<LOCATION>, <LOCATION>",788.0,230.0,es,Positivo,Positivo
1,04d09936041bc786,USR_22126242,[NOMBRE],[TWEET 1] RT [USUARIO]: Si ante la desgracia d...,100,2023-12-08,2026-04-16,"Psicóloga. Especializada en trauma, trastorn...",NaN,9186.0,351.0,es,Dudoso,Positivo
2,05306b7fa750a5d8,USR_e1786264,[NOMBRE],[TWEET 1] Si no dice Israel en el título no es...,100,2025-07-28,2026-04-20,"Orgullosamente judio. Si sos nazi, zurdo o k n...","<LOCATION>, <LOCATION>",8515.0,4776.0,es,Positivo,Positivo
3,0532b08c30376fde,USR_7ff3b9c0,[NOMBRE],[TWEET 1] llevo todo el día en la cama\n\n[TWE...,99,2024-01-21,2024-12-18,ig;:? ñ,"<LOCATION>, <LOCATION>",26.0,140.0,es,Positivo,Positivo
4,0588fd52a2609179,USR_8794a5e3,[NOMBRE],[TWEET 1] RT [USUARIO]: Optimus Christ [URL]\n...,98,2026-04-07,2026-04-20,Androide power!!! Obsesionado de la tecnología...,"<LOCATION>, <LOCATION>",709.0,3111.0,en,Dudoso,Positivo


In [7]:
tweets_filtrados = tweets_agrupados[tweets_agrupados['clasif_experto']=='Positivo']
tweets_filtrados.shape

(73, 14)

## **1. Instalación de dependencias**

In [ ]:
!pip install -q -U transformers>=4.50.0 accelerate bitsandbytes
!pip install -q langdetect sentencepiece sacremoses
!pip install -q tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 20.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 51.7 MB/s eta 0:00:00


## **2. Carga del modelo**

In [ ]:
# Gemma 3 12B (sin token, mirror Unsloth)
import torch, re, json, unicodedata, gc
from transformers import (
    AutoProcessor, Gemma3ForConditionalGeneration, BitsAndBytesConfig,
    MarianMTModel, MarianTokenizer,
)
from langdetect import detect, DetectorFactory, LangDetectException
from tqdm.auto import tqdm

# Determinismo en detección de idioma
DetectorFactory.seed = 42

GEMMA_MODEL_ID = "unsloth/gemma-3-12b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"⬇️  Descargando y cargando {GEMMA_MODEL_ID} (4-bit)...")
processor = AutoProcessor.from_pretrained(GEMMA_MODEL_ID)
model = Gemma3ForConditionalGeneration.from_pretrained(
    GEMMA_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.eval()
tokenizer = processor.tokenizer  # alias para usar eos_token_id
print("✅ Gemma 3 12B cargado.")


⬇️  Descargando y cargando unsloth/gemma-3-12b-it (4-bit)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

✅ Gemma 3 12B cargado.


## **3. Modelos de traducción Helsinki-NLP/opus-mt**

In [ ]:
TRANSLATION_MODELS = {}

# Idiomas soportados para traducir AL ESPAÑOL
SUPPORTED_LANGS_TO_ES = {"ca", "en", "pt", "it", "fr", "gl"}

def _load_translator(src: str, tgt: str):
    """Carga (y cachea) un par de traducción opus-mt."""
    key = f"{src}-{tgt}"
    if key in TRANSLATION_MODELS:
        return TRANSLATION_MODELS[key]
    model_name = f"Helsinki-NLP/opus-mt-{src}-{tgt}"
    try:
        tok = MarianTokenizer.from_pretrained(model_name)
        mdl = MarianMTModel.from_pretrained(model_name)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        mdl = mdl.to(device)
        mdl.eval()
        TRANSLATION_MODELS[key] = (tok, mdl)
        return tok, mdl
    except Exception as e:
        print(f"⚠️ No se pudo cargar {model_name}: {str(e)[:80]}")
        TRANSLATION_MODELS[key] = None
        return None

# Pre-cargar ES→FR (siempre se usa) y los traductores más comunes a ES
print("⬇️  Pre-cargando traductores...")
_load_translator("es", "fr")
_load_translator("en", "es")
_load_translator("ca", "es")
_load_translator("pt", "es")
print("✅ Traductores cargados.")

@torch.inference_mode()
def translate(text: str, src: str, tgt: str) -> str:
    """Traduce texto. Si falla, devuelve el texto original."""
    if not text or not text.strip():
        return text
    pair = _load_translator(src, tgt)
    if pair is None:
        return text
    tok, mdl = pair
    try:
        inputs = tok(text, return_tensors="pt", truncation=True, max_length=512).to(mdl.device)
        outputs = mdl.generate(**inputs, max_length=512, num_beams=4, early_stopping=True)
        return tok.decode(outputs[0], skip_special_tokens=True)
    except Exception as e:
        return text


⬇️  Pre-cargando traductores...


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/819k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.34M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/281M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/281M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

⚠️ No se pudo cargar Helsinki-NLP/opus-mt-pt-es: Helsinki-NLP/opus-mt-pt-es is not a local folder and is not a valid model identi
✅ Traductores cargados.


## **4. ETAPA 1 — Normalización Unicode**

In [ ]:
def normalize_unicode_text(text: str) -> str:
    """
    Convierte caracteres Unicode estilizados (Mathematical Alphanumeric,
    fullwidth, etc.) a su equivalente ASCII normal mediante NFKC.

    Ejemplos:
      '𝚗𝚎𝚐𝚊𝚝𝚒𝚟𝚎 𝚌𝚕𝚘𝚞𝚍' → 'negative cloud'
      '𝐸𝑥 𝑓𝑎𝑣𝑖𝑙𝑙𝑎'         → 'Ex favilla'
    """
    if not isinstance(text, str):
        return ""
    return unicodedata.normalize("NFKC", text)


## **5. ETAPA 2 — Detección de idioma + (opcional) traducción a ES**

In [ ]:
def detect_bio_language(text: str) -> str:
    """Detecta idioma del texto. Devuelve código ISO 639-1 o 'unknown'."""
    if not text or len(text.strip()) < 5:
        return "unknown"
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"

def ensure_spanish(text: str) -> tuple[str, str, bool]:
    """
    Devuelve (texto_en_es, idioma_detectado, fue_traducido).
    - Si es español: devuelve tal cual.
    - Si es idioma soportado: traduce a español.
    - Si es idioma raro / no soportado: devuelve tal cual con flag.
    """
    lang = detect_bio_language(text)
    if lang == "es" or lang == "unknown":
        return text, lang, False
    if lang in SUPPORTED_LANGS_TO_ES:
        translated = translate(text, lang, "es")
        return translated, lang, True
    # Idioma raro (turco, latín, etc.): no se traduce, pasa al LLM tal cual
    return text, lang, False


## **6. ETAPA 3 — LLM Generalizer con prompt género/edad-aware**

In [ ]:
GENERALIZER_PROMPT = """Eres un anonimizador de perfiles de Twitter para investigación en salud mental.

CONTEXTO: Tras tu reescritura, otro proceso intentará inferir GÉNERO y RANGO DE EDAD del usuario.
Tu output debe ANONIMIZAR identificadores específicos pero PRESERVAR señales demográficas útiles.

═══════════ ELIMINA SI APARECE ═══════════
- Nombres propios de personas, hijos, parejas, mascotas
- Organizaciones específicas (asociaciones, fundaciones, medios, hospitales, partidos políticos, marcas)
- Ubicaciones específicas (ciudades, barrios, provincias, países)
- Handles de redes sociales aunque no lleven @ (Instagram, TikTok, Snapchat, etc.)
- Fechas exactas (DD/MM/AAAA o años específicos)
- Combinaciones únicas (cargo + organización + ubicación)
- Marcadores residuales: <LOCATION>, <PERSON>, <PHONE_NUMBER>, [USUARIO], [URL], [EMAIL], [NUMERO], [NOMBRE], [PERSONA], [UBICACION]
  → DEBES ELIMINARLOS de tu output. No los reescribas tal cual.

═══════════ PRESERVA SIEMPRE QUE ESTÉ ═══════════
- Pronombres y marcas de género (She/Her, He/Him, They/Them, Ella/Él)
- Género gramatical de adjetivos en español (psicóloga vs psicólogo)
- Roles familiares genéricos (madre, padre, hijo, abuela) sin nombres
- Categoría profesional genérica (profesional sanitario, jurista, docente, ingeniero)
- Indicadores etarios (generación Z, "23 años", "nacido en los 90", aproximaciones)
- Región amplia (España, Latinoamérica) si era explícita
- Temas de interés generales (música, deporte, salud mental, política, gaming, anime)

═══════════ CASOS ESPECIALES — DEVUÉLVELA TAL CUAL ═══════════
- Es solo emojis: "🩷", "🏳️‍🌈", "🇪🇦"
- Es una frase poética sin info personal: "Aprendiendo a vivir ✨"
- Es ruido o interjección sin contenido: "khé", "pa yaso", "ni idea bro"

═══════════ EJEMPLOS ═══════════

ORIGINAL: "Psicóloga. Especializada en trauma, trastornos de la conducta alimentaria y apego."
GENERALIZADA: "Mujer profesional de la psicología clínica con especialización en salud mental."

ORIGINAL: "Andalú sin pareserlo. Cocino mucho en Instagram 365diascomiendo. Cocinero, Dietista y costurero."
GENERALIZADA: "Hombre del sur de España con interés en gastronomía y nutrición, activo en redes sociales."

ORIGINAL: "💔PADRE DE KIRA💔 Perito Judicial en Acoso Escolar. Presidente de Trencats Asociación contra la Violencia en las Escuelas."
GENERALIZADA: "Hombre adulto, padre, profesional jurídico activo en una causa social relacionada con violencia escolar."

ORIGINAL: "Orgullosamente judio. Sionista, ahora en <LOCATION>. Viajar es la mejor inversión"
GENERALIZADA: "Persona de fe judía con identidad sionista, interés en viajar."

ORIGINAL: "23. amor ilegal instagram: mumi__sanchez"
GENERALIZADA: "Persona joven, aproximadamente veintitantos años, activa en redes sociales."

ORIGINAL: "Mamá de <PERSON> | Extremeña y Madrileña"
GENERALIZADA: "Mujer adulta, madre, vinculada al centro y oeste de España."

ORIGINAL: "🩷"
GENERALIZADA: "🩷"

ORIGINAL: "Aprendiendo a vivir ✨"
GENERALIZADA: "Aprendiendo a vivir ✨"

ORIGINAL: "Androide power!!! Obsesionado de la tecnología y las consolas. Nintendero desde 1993"
GENERALIZADA: "Persona aficionada a la tecnología y los videojuegos, con interés desde los años 90."

BIO A GENERALIZAR: "{bio}"
GENERALIZADA:"""


@torch.inference_mode()
def generalize_bio_llm(bio_text: str) -> str:
    """Llama a Gemma 3 12B con el prompt género/edad-aware."""
    if not bio_text or not bio_text.strip():
        return bio_text

    safe_prompt = GENERALIZER_PROMPT.replace("{bio}", bio_text)
    messages = [{"role": "user", "content": [{"type": "text", "text": safe_prompt}]}]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    gen_tokens = outputs[0][input_len:]
    response = processor.decode(gen_tokens, skip_special_tokens=True).strip()

    # Limpiar prefijos típicos que el modelo a veces reproduce
    response = re.sub(r"^(GENERALIZADA|GEN|RESPONSE)[:\s]*", "", response, flags=re.IGNORECASE).strip()
    # Quitar comillas envolventes
    response = response.strip('"\'')
    # Si el modelo devolvió varias líneas: quedarse con la primera no vacía
    lines = [l.strip() for l in response.split("\n") if l.strip()]
    if lines:
        response = lines[0]
    return response

## **7. ETAPA 4 — Validadores SOFT (reportan, no bloquean)**

In [ ]:
SENALES_GENERO_REGEX = [
    (r"\b(she|her|hers)\b",                    "pronombre_femenino_en"),
    (r"\b(he|him|his)\b",                      "pronombre_masculino_en"),
    (r"\b(they|them|their)\b",                 "pronombre_no_binario_en"),
    (r"\bella\b",                              "pronombre_femenino_es"),
    (r"\b(mam[áa]|madre|mujer|chica|señora|abuela|hija)\b", "femenino_implicito"),
    (r"\b(pap[áa]|padre|hombre|chico|señor|abuelo|hijo|varón)\b", "masculino_implicito"),
    (r"\b(trans|transgénero|transexual)\b",    "identidad_trans"),
    (r"\b(they/them|he/him|she/her|she/ella|he/el)\b", "pronombre_explicito"),
    (r"\b(no binari[ao]|nonbinary)\b",          "no_binario"),
    # Heurística adjetivo terminado en -a/-o + verbo "soy/estoy"
    (r"\b\w+a\b\s+(soy|estoy|orgullos)",     "femenino_gramatical"),
    (r"\b\w+o\b\s+(soy|estoy|orgullos)",     "masculino_gramatical"),
]

SENALES_EDAD_REGEX = [
    (r"\b\d{1,2}\s*(años|tacos|primaveras)\b", "edad_explicita"),
    (r"\b(generaci[óo]n\s*[zxy])\b",            "generacion"),
    (r"\bdesde\s+(19|20)\d{2}\b",              "desde_año"),
    (r"\b(naci[óo]\s+en\s+(19|20)\d{2})\b",   "nacido_en_año"),
    (r"\b(joven|adolescente|adulto|mayor|s[ée]nior|jubilad[ao])\b", "rango_generico"),
    (r"\b(boomer|millenial|millennial|zoomer|gen[ -]?z|gen[ -]?x)\b", "jerga_generacional"),
    (r"\b(años?\s*90|años?\s*80|años?\s*2000)\b", "decada"),
]

def extract_signals(text: str, regex_list) -> list:
    if not text:
        return []
    found = []
    for pattern, label in regex_list:
        if re.search(pattern, text, flags=re.IGNORECASE):
            found.append(label)
    return found

def validate_anonymization(original: str, generalized: str) -> list:
    """Validación dura: PII residual o problemas técnicos."""
    issues = []
    if not generalized or not generalized.strip():
        issues.append("output_vacio")
        return issues
    
    # Copia literal (señal de que el modelo no generalizó)
    if original.strip() and original.strip() == generalized.strip() and len(original) > 60:
        issues.append("copia_literal_larga")

    # Placeholders residuales
    placeholders = ["<LOCATION>", "<PERSON>", "<PHONE_NUMBER>",
                    "[USUARIO]", "[URL]", "[EMAIL]", "[NUMERO]",
                    "[NOMBRE]", "[PERSONA]", "[UBICACION]"]
    for ph in placeholders:
        if ph in generalized:
            issues.append(f"placeholder_residual:{ph}")

    # Longitud excesiva
    if len(generalized) > 500:
        issues.append("muy_largo")
    return issues

def validate_demographic_signals_soft(original: str, generalized: str) -> dict:
    """
    Validación blanda: SOLO reporta lo que el LLM preservó o perdió.
    No bloquea. El paso posterior (inferencia género/edad) decidirá qué bios usar.
    """
    genero_orig = extract_signals(original, SENALES_GENERO_REGEX)
    genero_gen  = extract_signals(generalized, SENALES_GENERO_REGEX)
    edad_orig   = extract_signals(original, SENALES_EDAD_REGEX)
    edad_gen    = extract_signals(generalized, SENALES_EDAD_REGEX)
    return {
        "genero_signals_orig":   genero_orig,
        "genero_signals_gen":    genero_gen,
        "edad_signals_orig":     edad_orig,
        "edad_signals_gen":      edad_gen,
        "genero_preservado":     (bool(genero_gen) if genero_orig else None),
        "edad_preservado":       (bool(edad_gen) if edad_orig else None),
    }

## **8. ETAPA 5 — Traducción ES → FR (universal)**

In [ ]:
def translate_to_french(text: str) -> str:
    """Traduce del español al francés con opus-mt-es-fr."""
    return translate(text, "es", "fr")

## **9. ORQUESTADOR — pipeline completo para una bio**

In [ ]:
import pandas as pd

def process_bio(bio_raw) -> dict:
    """Aplica las 5 etapas a una bio. Devuelve registro completo de auditoría."""
    audit = {
        "bio_original":   bio_raw if isinstance(bio_raw, str) else str(bio_raw),
        "es_nan":         False,
        "es_vacia":       False,
    }

    # Manejo de NaN / no-string
    if bio_raw is None or (isinstance(bio_raw, float) and pd.isna(bio_raw)):
        audit["es_nan"] = True
        audit["bio_final_fr"] = ""
        return audit
    if not isinstance(bio_raw, str) or bio_raw.strip().lower() in ("nan", "none", ""):
        audit["es_nan"] = True
        audit["bio_final_fr"] = ""
        return audit

    # Etapa 1: Normalización Unicode
    bio_norm = normalize_unicode_text(bio_raw)
    audit["bio_normalizada"] = bio_norm

    if len(bio_norm.strip()) < 3:
        audit["es_vacia"] = True
        audit["bio_final_fr"] = bio_norm
        return audit

    # Etapa 2: Detección de idioma + traducción a ES (si aplica)
    bio_es, lang, fue_traducida = ensure_spanish(bio_norm)
    audit["idioma_detectado"] = lang
    audit["traducida_a_es"]   = fue_traducida
    audit["flag_idioma_raro"] = (lang != "es" and lang != "unknown" and lang not in SUPPORTED_LANGS_TO_ES)
    if fue_traducida:
        audit["bio_es"] = bio_es

    # Etapa 3: LLM Generalizer
    try:
        bio_general = generalize_bio_llm(bio_es)
    except Exception as e:
        audit["error_llm"] = str(e)[:120]
        bio_general = bio_es  # fallback al texto en español sin generalizar
    audit["bio_generalizada_es"] = bio_general

    # Etapa 4: Validaciones (soft)
    audit["validacion_anonimizacion"] = validate_anonymization(bio_es, bio_general)
    audit["validacion_demografica"]   = validate_demographic_signals_soft(bio_es, bio_general)

    # Etapa 5: Traducción ES → FR
    try:
        bio_fr = translate_to_french(bio_general)
    except Exception as e:
        audit["error_traduccion_fr"] = str(e)[:120]
        bio_fr = bio_general  # fallback
    audit["bio_final_fr"] = bio_fr

    return audit

## **10. Ejecución**

In [ ]:
# Recorre tweets_filtrados['profile_bio']

# Requiere que el DataFrame `tweets_filtrados` ya esté cargado en memoria
assert "tweets_filtrados" in dir(), "❌ Carga primero el DataFrame `tweets_filtrados` antes de esta celda."
assert "profile_bio" in tweets_filtrados.columns, "❌ El DataFrame no tiene columna `profile_bio`."

total = len(tweets_filtrados)
print(f"🚀 Procesando {total} bios...")

audit_log = []
bios_fr   = []

for idx, bio in enumerate(tqdm(tweets_filtrados["profile_bio"], desc="Anonimizando bios", unit="bio")):
    try:
        result = process_bio(bio)
    except Exception as e:
        result = {
            "bio_original": str(bio)[:200],
            "error_critico": str(e)[:200],
            "bio_final_fr": "",
        }
    audit_log.append(result)
    bios_fr.append(result.get("bio_final_fr", ""))

print("\n✅ Procesamiento completo.")

# Añadir la columna nueva al DataFrame
tweets_filtrados["profile_bio_fr"] = bios_fr


🚀 Procesando 73 bios...


Anonimizando bios:   0%|          | 0/73 [00:00<?, ?bio/s]

[transformers] Deprecated: `processor.image_token` will switch from returning `tokenizer.image_token` to `tokenizer.boi_token` in v5.11.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/813k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/300M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/300M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]


✅ Procesamiento completo.


In [21]:
tweets_filtrados = tweets_filtrados.drop(columns=['profile_bio'])

In [23]:
tweets_filtrados.to_csv('/content/tweets_filtrados_final.csv', index=False)
print("DataFrame 'tweets_filtrados' exportado a '/content/tweets_filtrados_final.csv'")

DataFrame 'tweets_filtrados' exportado a '/content/tweets_filtrados_final.csv'


## **11. Guardado + Resumen de auditoría**

In [ ]:
# Dataset enriquecido
output_csv = "/content/tweets_filtrados_bios_fr.csv"
tweets_filtrados.to_csv(output_csv, index=False)
print(f"💾 DataFrame guardado: {output_csv}")

# Auditoría completa
output_audit = "/content/bio_pipeline_audit.json"
with open(output_audit, "w", encoding="utf-8") as f:
    json.dump(audit_log, f, ensure_ascii=False, indent=2)
print(f"💾 Auditoría guardada: {output_audit}")

# Resumen agregado
print("\n" + "="*60)
print("📊 RESUMEN DEL PIPELINE")
print("="*60)
total = len(audit_log)
nan_o_vacia = sum(1 for a in audit_log if a.get("es_nan") or a.get("es_vacia"))
traducidas = sum(1 for a in audit_log if a.get("traducida_a_es"))
idioma_raro = sum(1 for a in audit_log if a.get("flag_idioma_raro"))
errores_llm = sum(1 for a in audit_log if a.get("error_llm"))
errores_fr  = sum(1 for a in audit_log if a.get("error_traduccion_fr"))
errores_crit = sum(1 for a in audit_log if a.get("error_critico"))

print(f"  Total bios:                {total}")
print(f"  NaN / vacías:              {nan_o_vacia}")
print(f"  Traducidas a ES intermedio:{traducidas}")
print(f"  Idioma raro (sin traducir): {idioma_raro}")
print(f"  Errores en LLM:            {errores_llm}")
print(f"  Errores en traducción FR:  {errores_fr}")
print(f"  Errores críticos:          {errores_crit}")

# Métricas de validación soft (informativas)
genero_perdido = sum(1 for a in audit_log
                     if a.get("validacion_demografica", {}).get("genero_preservado") is False)
edad_perdida = sum(1 for a in audit_log
                   if a.get("validacion_demografica", {}).get("edad_preservado") is False)
print(f"\n  Bios con género preservado: {sum(1 for a in audit_log if a.get('validacion_demografica', {}).get('genero_preservado') is True)}")
print(f"  Bios donde se perdió señal de género: {genero_perdido}")
print(f"  Bios donde se perdió señal de edad:   {edad_perdida}")

# Anonimización: cuántas tuvieron issues
anon_issues = sum(1 for a in audit_log if a.get("validacion_anonimizacion"))
print(f"  Bios con issues de anonimización:     {anon_issues}")
print("="*60)


💾 DataFrame guardado: /content/tweets_filtrados_bios_fr.csv
💾 Auditoría guardada: /content/bio_pipeline_audit.json

📊 RESUMEN DEL PIPELINE
  Total bios:                73
  NaN / vacías:              8
  Traducidas a ES intermedio:19
  Idioma raro (sin traducir): 10
  Errores en LLM:            0
  Errores en traducción FR:  0
  Errores críticos:          0

  Bios con género preservado: 4
  Bios donde se perdió señal de género: 2
  Bios donde se perdió señal de edad:   1
  Bios con issues de anonimización:     3


## **12. Inspección rápida**

In [ ]:
# Primeras 10 bios procesadas
for i, a in enumerate(audit_log[:10]):
    print(f"\n──── Bio #{i} ────")
    print(f"  Original:     {a.get('bio_original', '')[:120]}")
    if a.get("es_nan"):
        print("  [NaN/vacía — sin procesar]")
        continue
    print(f"  Idioma:       {a.get('idioma_detectado')}")
    if a.get("traducida_a_es"):
        print(f"  Traducida ES: {a.get('bio_es', '')[:120]}")
    print(f"  Generalizada: {a.get('bio_generalizada_es', '')[:120]}")
    print(f"  Final FR:     {a.get('bio_final_fr', '')[:120]}")
    if a.get("validacion_anonimizacion"):
        print(f"  Issues anon:  {a.get('validacion_anonimizacion')}")
    dem = a.get("validacion_demografica", {})
    if dem.get("genero_signals_orig"):
        print(f"  Género orig→gen: {dem['genero_signals_orig']} → {dem['genero_signals_gen']}")
    if dem.get("edad_signals_orig"):
        print(f"  Edad orig→gen:   {dem['edad_signals_orig']} → {dem['edad_signals_gen']}")



──── Bio #0 ────
  Original:     nan
  [NaN/vacía — sin procesar]

──── Bio #1 ────
  Original:     Psicóloga.   Especializada en trauma, trastornos de la conducta alimentaria y apego.
  Idioma:       es
  Generalizada: Mujer profesional de la psicología clínica con especialización en salud mental.
  Final FR:     Femme professionnelle de la psychologie clinique spécialisée dans la santé mentale.

──── Bio #2 ────
  Original:     Orgullosamente judio. Si sos nazi, zurdo o k no te quiero.
Si ellos son la patria yo soy extranjero.
Sionista, ahora en 
  Idioma:       es
  Generalizada: Persona de fe judía con identidad sionista, interés en viajar.
  Final FR:     Personne de foi juive avec une identité sioniste, intérêt à voyager.
  Género orig→gen: ['masculino_gramatical'] → []

──── Bio #3 ────
  Original:     ig;:? ñ
  Idioma:       tl
  Generalizada: Persona activa en redes sociales.
  Final FR:     Personne active sur les réseaux sociaux.

──── Bio #4 ────
  Original:     Androide p